<a href="https://colab.research.google.com/github/FrauMeka/homework5/blob/main/Hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dataclasses import dataclass
from datetime import datetime
import re


def identify_applicable_regulations(org_profile: dict) -> list:

    regulations = []
    sector = org_profile.get("sector", "").strip().lower()
    employees = org_profile.get("employees", 0)

    def add(name: str, reason: str, requirement: str,
            priority: str = "MANDATORY") -> None:
        regulations.append({"name": name, "reason": reason,
                            "priority": priority, "key_requirement": requirement})

    if org_profile.get("handles_health_data", False) or sector in {
        "healthcare", "hospital", "clinic"
    }:
        add("HIPAA Security Rule", "Handles protected health information (PHI).",
            "Risk analysis and administrative, physical and technical safeguards.")
    if org_profile.get("processes_payments", False):
        add("PCI-DSS v4.0", "Processes, stores or transmits cardholder data.",
            "Payment data protection and an appropriate assessment or SAQ.")
    if org_profile.get("handles_eu_personal_data", False):
        add("GDPR", "Processes EU personal data within the assignment scope.",
            "Lawful processing, data subject rights and breach notification.")
    if sector in {"banking", "financial services", "insurance", "fintech"}:
        add("GLBA Safeguards Rule", "Financial institution in the course model.",
            "Written information security program and a qualified responsible person.")
    if org_profile.get("public_company", False):
        add("SOX IT Controls (Section 404)", "Publicly traded company in the course model.",
            "Assessment of internal controls over financial reporting.")
    if org_profile.get("is_federal_contractor", False) or sector == "government":
        add("FISMA / NIST SP 800-53", "Federal agency or contractor in the course model.",
            "Security controls, authorization and continuous monitoring.")
    if org_profile.get("handles_cui", False) or sector == "defense":
        add("CMMC 2.0", "Defense contracting or controlled unclassified information.",
            "Assess the required CMMC level and applicable contract requirements.")
    if org_profile.get("california_presence", False) and employees >= 1:
        add("CCPA/CPRA", "California presence; applicability thresholds need checking.",
            "Verify thresholds; support applicable consumer privacy rights.",
            "LIKELY APPLICABLE - verify thresholds")
    add("NIST Cybersecurity Framework 2.0", "Voluntary baseline for any organization.",
        "Govern, Identify, Protect, Detect, Respond and Recover.",
        "STRONGLY RECOMMENDED")
    return regulations


# The weights reproduce the instructor's maturity calculation.
MATURITY_RULES = {
    "governance": [
        ("has_ciso", 1, "CISO or dedicated security leader exists"),
        ("has_security_committee", 1, "Security/risk committee established"),
        ("board_security_reporting", 1, "Board receives regular security reports"),
        ("security_budget_defined", 0.5, "Security budget is defined"),
        ("policy_suite_current", 0.5, "Policies reviewed within 12 months"),
    ],
    "risk": [
        ("has_risk_register", 1, "Formal risk register maintained"),
        ("annual_risk_assessment", 1, "Annual risk assessment conducted"),
        ("continuous_risk_monitoring", 1, "Continuous risk monitoring in place"),
        ("risk_appetite_documented", 1, "Risk appetite documented and approved"),
    ],
    "compliance": [
        ("compliance_program", 1, "Formal compliance program exists"),
        ("last_audit_passed", 1, "Most recent audit passed"),
        ("security_training", 1, "Security awareness training is active"),
        ("vendor_risk_program", 1, "Vendor risk assessment program exists"),
    ],
}


def assess_governance_maturity(org_profile: dict) -> dict:
    """Calculate 1–5 scores, positive evidence and their rounded average.

    Each dimension starts at 1 and gains points for existing controls. Missing
    values count as no confirmed evidence, not proof that a control is absent.
    """
    scores = {}
    evidence = {}
    for dimension, rules in MATURITY_RULES.items():
        score = 1
        evidence[dimension] = []
        for field, weight, description in rules:
            if org_profile.get(field, False):
                score += weight
                evidence[dimension].append("+ " + description)
        scores[dimension] = min(5, round(score, 1))
    return {
        "scores": scores,
        "evidence": evidence,
        "overall_maturity": round(sum(scores.values()) / 3, 1),
    }


def generate_grc_summary_report(org_profile: dict) -> str:
    """Combine framework screening and maturity evidence in a readable report."""
    regulations = identify_applicable_regulations(org_profile)
    maturity = assess_governance_maturity(org_profile)
    lines = [
        "=" * 78,
        "GRC READINESS ASSESSMENT REPORT",
        f"Organization: {org_profile.get('org_name', 'Unknown')}",
        f"Sector: {org_profile.get('sector', 'Unknown')}",
        f"Employees: {org_profile.get('employees', 0)}",
        f"Generated: {datetime.now():%Y-%m-%d %H:%M}",
        "Classroom screening model: verify actual regulatory scope separately.",
        "=" * 78,
        "", "APPLICABLE REGULATORY FRAMEWORKS",
    ]
    for regulation in regulations:
        lines.extend([
            f"[{regulation['priority']}] {regulation['name']}",
            f"  Reason: {regulation['reason']}",
            f"  Key requirement: {regulation['key_requirement']}", "",
        ])
    lines.append("GRC MATURITY SCORES (1 = Initial; 5 = Optimizing)")
    for dimension, score in maturity["scores"].items():
        lines.append(f"{dimension.upper():<12}: {score:.1f}/5.0")
        lines.extend("  " + item for item in maturity["evidence"][dimension])
        if not maturity["evidence"][dimension]:
            lines.append("  No positive evidence recorded in the profile.")
    overall = maturity["overall_maturity"]
    status = ("CRITICAL GAPS - IMMEDIATE ACTION NEEDED" if overall < 2 else
              "DEVELOPING - SIGNIFICANT IMPROVEMENT NEEDED" if overall < 3 else
              "DEFINED - CONTINUE BUILDING CONSISTENCY" if overall < 4 else
              "MANAGED - FOCUS ON OPTIMIZATION" if overall < 5 else
              "OPTIMIZING - MAINTAIN AND REVIEW CONTROLS")
    lines.extend(["", f"OVERALL GRC MATURITY: {overall:.1f}/5.0", status])
    lines.extend(["", "PRIORITY IMPROVEMENTS"])
    for dimension, rules in MATURITY_RULES.items():
        for field, _, description in rules:
            if not org_profile.get(field, False):
                lines.append(f"  {dimension.upper()}: establish or verify: {description}.")
    lines.append("=" * 78)
    return "\n".join(lines)


@dataclass
class SecurityEvent:
    """A security incident with a name and a short description."""

    name: str
    description: str


def classify_cia_impact(event: SecurityEvent) -> dict:
    """Classify CIA impact with the course's keyword model and priority order.

    Whole-word prefixes prevent 'unencrypted' matching 'encrypt' and 'thread'
    matching 'read'. Priority is availability, confidentiality, then integrity.
    Severity counts affected CIA dimensions; it is not actual incident severity.
    The model cannot reliably interpret negation or every possible incident.
    """
    text = f"{event.name} {event.description}".lower()
    confidentiality = bool(re.search(
        r"\b(?:data exposed|leaked|unauthorized access|breach\w*|exfiltrat\w*|"
        r"credential\w*|eavesdrop\w*|intercept\w*|disclosed|stolen|read|"
        r"viewed by unauthorized)\b", text))
    integrity = bool(re.search(
        r"\b(?:modified|tamper\w*|altered|corrupt\w*|falsif\w*|changed without|"
        r"forged|injected|replaced|sql inject\w*|man-in-the-middle)\b", text))
    availability = bool(re.search(
        r"\b(?:unavailable|down|outage|ransomware|encrypt\w*|ddos|"
        r"denial of service|locked out|deleted|wiped|destroyed|inaccessible)\b", text))
    if availability:
        primary = "AVAILABILITY"
    elif confidentiality:
        primary = "CONFIDENTIALITY"
    elif integrity:
        primary = "INTEGRITY"
    else:
        primary = "UNCLEAR - manual review needed"
    severity = max(1, sum([confidentiality, integrity, availability]))
    return {
        "event": event.name,
        "confidentiality_affected": confidentiality,
        "integrity_affected": integrity,
        "availability_affected": availability,
        "primary_impact": primary,
        "severity": severity,
        "severity_label": {1: "LOW", 2: "MEDIUM", 3: "HIGH"}[severity],
    }


def print_cia_report(events: list[SecurityEvent]) -> None:
    """Print every event with the C, I, A, PRIMARY and SEV columns."""
    results = [classify_cia_impact(event) for event in events]
    name_width = max([len("EVENT")] + [len(event.name) for event in events])
    primary_width = max([len("PRIMARY")] + [len(r["primary_impact"]) for r in results])
    width = name_width + primary_width + 24
    print("\n" + "=" * width)
    print("CIA TRIAD IMPACT CLASSIFICATION REPORT")
    print("=" * width)
    print(f"{'EVENT':<{name_width}}  C  I  A  {'PRIMARY':<{primary_width}}  SEV")
    print("-" * width)
    for result in results:
        flags = ["Y" if result[key] else "X" for key in (
            "confidentiality_affected", "integrity_affected", "availability_affected")]
        print(f"{result['event']:<{name_width}}  {'  '.join(flags)}  "
              f"{result['primary_impact']:<{primary_width}}  {result['severity_label']}")
    print("=" * width)
    print("C=Confidentiality  I=Integrity  A=Availability; Y=affected, X=not detected")
    print("SEV is the classroom CIA-count label, not a real risk rating.")


# Instructor's eight test cases: names and descriptions are preserved.
TEST_EVENTS = [
    SecurityEvent("HealthBridge PHI Breach", "Attacker used credential stolen via spyware; 14,200 patient records data exposed to unauthorized party"),
    SecurityEvent("Hospital Ransomware Attack", "All systems encrypted by ransomware; EHR unavailable for 5 days"),
    SecurityEvent("EHR Record Tampering", "Nurse altered medication dosage records in EHR; patient harmed"),
    SecurityEvent("DDoS on Bank Portal", "Customer online banking portal unavailable for 6 hours via DDoS"),
    SecurityEvent("SQL Injection Attack", "Attacker injected SQL into web form; customer records altered"),
    SecurityEvent("Laptop Theft", "Unencrypted laptop with 3,000 employee SSNs stolen from car"),
    SecurityEvent("Insider Data Exfiltration", "Employee emailed 50,000 customer records to personal email before resignation"),
    SecurityEvent("Power Outage", "Data center power outage; no UPS; servers unavailable 3 hours"),
]


# Instructor's HealthBridge test profile, preserved from the handout.
HEALTHBRIDGE = {
    "org_name": "HealthBridge Medical Center", "sector": "healthcare",
    "employees": 2800, "handles_health_data": True, "processes_payments": True,
    "handles_eu_personal_data": False, "public_company": False,
    "is_federal_contractor": False, "handles_cui": False,
    "california_presence": False, "has_ciso": False,
    "has_security_committee": False, "board_security_reporting": False,
    "security_budget_defined": False, "policy_suite_current": False,
    "has_risk_register": False, "annual_risk_assessment": True,
    "continuous_risk_monitoring": False, "risk_appetite_documented": False,
    "compliance_program": True, "last_audit_passed": True,
    "security_training": False, "vendor_risk_program": False,
}


# PROPOSED LAB 1 CASE — fictional, all details below are assumptions.
# Aurora Online Store is a US online retailer with 35 employees. In this case
# it directly processes cardholder data and handles EU customer personal data
# within GDPR scope. It has a security budget and current policies, but no
# security leader, committee or board reporting. It conducts annual risk
# assessments and security training, but lacks the other listed programs.
# Use this case for Lab 1 only if permitted; otherwise replace this dictionary.
MY_COMPANY = {
    "org_name": "Aurora Online Store (fictional proposed Lab 1 case)",
    "sector": "retail", "employees": 35,
    "handles_health_data": False, "processes_payments": True,
    "handles_eu_personal_data": True, "public_company": False,
    "is_federal_contractor": False, "handles_cui": False,
    "california_presence": False, "has_ciso": False,
    "has_security_committee": False, "board_security_reporting": False,
    "security_budget_defined": True, "policy_suite_current": True,
    "has_risk_register": False, "annual_risk_assessment": True,
    "continuous_risk_monitoring": False, "risk_appetite_documented": False,
    "compliance_program": False, "last_audit_passed": False,
    "security_training": True, "vendor_risk_program": False,
}


if __name__ == "__main__":
    # Main output uses the student's proposed company rather than HealthBridge.
    print(generate_grc_summary_report(MY_COMPANY))
    print_cia_report(TEST_EVENTS)


GRC READINESS ASSESSMENT REPORT
Organization: Aurora Online Store (fictional proposed Lab 1 case)
Sector: retail
Employees: 35
Generated: 2026-10-05 05:33
Classroom screening model: verify actual regulatory scope separately.

APPLICABLE REGULATORY FRAMEWORKS
[MANDATORY] PCI-DSS v4.0
  Reason: Processes, stores or transmits cardholder data.
  Key requirement: Payment data protection and an appropriate assessment or SAQ.

[MANDATORY] GDPR
  Reason: Processes EU personal data within the assignment scope.
  Key requirement: Lawful processing, data subject rights and breach notification.

[STRONGLY RECOMMENDED] NIST Cybersecurity Framework 2.0
  Reason: Voluntary baseline for any organization.
  Key requirement: Govern, Identify, Protect, Detect, Respond and Recover.

GRC MATURITY SCORES (1 = Initial; 5 = Optimizing)
GOVERNANCE  : 2.0/5.0
  + Security budget is defined
  + Policies reviewed within 12 months
RISK        : 2.0/5.0
  + Annual risk assessment conducted
COMPLIANCE  : 2.0/5.0
  +